In [ ]:
%load_ext autoreload
%autoreload 2

import yaml
import polars as pl
import numpy as np
import torch

from anngeno import AnnGeno
from scripts import get_burdens, get_correlations

device = "cuda" if torch.cuda.is_available() else "cpu"
device

## Read Anngeno file

In [ ]:
anngeno_path = "/home/dnanexus/data_dir/anngeno_training.ag"
ag = AnnGeno(anngeno_path,  filemode="r", low_mem=True)
ag


In [ ]:
ag.phenotypes

## Read gene-trait associations

In [ ]:
gt = pl.read_parquet("/home/dnanexus/genebass_continuous_associations_ukbbgym.pq")[['gene_id', 'gene_symbol', 'description']].unique()

gt = gt.with_columns(pl.col("description").str.to_lowercase().alias("description"))
gt = gt.with_columns(pl.col("description").str.replace_all("-", "").alias("description"))
gt = gt.with_columns(pl.col("description").str.replace_all(" ", "_").alias("phenotype")).drop(['description'])
gt

### Subset associations to genes and phenoptypes available in the small Anngeno

In [ ]:
subsest_gt = gt.filter(
    pl.col("gene_id").is_in(ag.annotations.select(pl.col("region")).collect().unique()['region'])
    ).filter(
    pl.col("phenotype").is_in(ag.phenotypes.columns)
    )

subsest_gt

### Small subset of associations to sanity check

In [ ]:
subsest_gt.filter(pl.col('gene_symbol').str.contains("LDLR")).write_parquet("/home/dnanexus/ldlr_genes.pq")


## Compute burdens for the small subset

In [ ]:
config_path = "/home/dnanexus/ukbgym/config_wgs.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)

all_annotation_list = []
rare_variant_annotations_dict = config.get('rare_variant_annotations')
if rare_variant_annotations_dict:
    for category in rare_variant_annotations_dict.values():
        all_annotation_list.extend(category)

all_annotation_list

In [ ]:
anngeno_path = "/home/dnanexus/data_dir/anngeno_training.ag"
associations_df_path = '/home/dnanexus/ldlr_genes.pq'

zarr_burdens_path = '/home/dnanexus/ukbgym/temp.zarr'

get_burdens.get_burdens_array(
    anngeno_path=anngeno_path,
    associations_df_path=associations_df_path,
    maf=0.001,
    annotation_list=[],
    max_burden=True,
    only_snps=False,
    n_jobs=1,
    batch_size=4,
    device=device,
)


## Phenotype GIS plot